# 💊 01 · Clean Drug Data — บัญชียาหลักแห่งชาติ

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nuttakarnsinpho610-ai/DADS5001_FINAL_PORJECT/blob/main/notebooks/01_clean_drug_data.ipynb)

**เป้าหมายของ notebook นี้:**
อ่านไฟล์ต้นฉบับจากบัญชียาหลักแห่งชาติ (`.xls` จากกระทรวงสาธารณสุข) → clean ข้อมูล → เพิ่มความหมายของหมวด ED + map กับ ICD-10 → save เป็น CSV ที่แอป Streamlit ใช้ต่อ

**ที่มาของข้อมูล:**
- 📂 บัญชียาหลักแห่งชาติ พ.ศ. 2558
- 🏛️ Publisher: กลุ่มนโยบายแห่งชาติด้านยา สำนักยา กระทรวงสาธารณสุข
- 🔗 URL: http://drug.fda.moph.go.th:81/nlem.in.th/medicine/essential/list
- 📜 License: Open Government License - Thailand

**ขั้นตอนใน notebook:**
1. ติดตั้ง library + import
2. อ่านไฟล์ `.xls` จาก GitHub (raw URL)
3. ตรวจสอบข้อมูลดิบ
4. Clean ช่องว่าง (whitespace)
5. เพิ่มความหมายหมวด ED + reimbursement note
6. Map กลุ่มยา → ICD-10 chapter (สำหรับเชื่อมกับฐาน 41 โรคของพี่ Getto)
7. ลบคอลัมน์ที่ไม่ใช้ + **เก็บข้อมูล clinical 4 คอลัมน์** (คำแนะนำ, เงื่อนไข, คำเตือน, หมายเหตุ)
8. Save เป็น CSV
9. ดาวน์โหลด CSV (สำหรับ upload กลับไป GitHub)

> 👤 **Author:** Kade · DADS5001 Final Project · Team พี่ Getto + Kade

## Step 1 · ติดตั้ง library + import

ไฟล์ `.xls` (ไม่ใช่ `.xlsx`) ต้องใช้ library ชื่อ `xlrd` ในการอ่าน — Colab ไม่ได้มาพร้อมโดย default ต้องลงเพิ่ม

In [ ]:
!pip install xlrd --quiet

import pandas as pd
import numpy as np

print("✅ pandas version:", pd.__version__)
print("✅ พร้อมใช้งาน")

## Step 2 · ตั้งค่า URL ของไฟล์ต้นฉบับใน GitHub

อ่านไฟล์โดยตรงจาก GitHub repo ของ Kade — ไม่ต้องอัพโหลดไฟล์ใส่ Colab ทุกครั้ง

In [ ]:
# === ข้อมูล GitHub repo ===
GITHUB_USER = "nuttakarnsinpho610-ai"
REPO_NAME   = "DADS5001_FINAL_PORJECT"
BRANCH      = "main"
FILE_PATH   = "data/raw/ed55-publish_post_web56_edit16-10-56.xls"

# Raw URL (สำหรับให้ pandas อ่านโดยตรง)
RAW_URL = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO_NAME}/{BRANCH}/{FILE_PATH}"
print("📥 จะอ่านจาก:", RAW_URL)

## Step 3 · อ่านไฟล์ `.xls` ต้นฉบับ

ในไฟล์ `.xls` มี 3 sheets: `CODE`, `edit rachakitcha 56`, `comment`

ใช้ sheet `edit rachakitcha 56` เพราะเป็น sheet ที่มีรายการยาทั้งหมด

In [ ]:
df_raw = pd.read_excel(RAW_URL, sheet_name="edit rachakitcha 56", header=0)

print(f"📊 ขนาดข้อมูลดิบ: {df_raw.shape[0]} แถว × {df_raw.shape[1]} คอลัมน์")
print(f"\n📋 ชื่อคอลัมน์ทั้งหมด:")
for i, c in enumerate(df_raw.columns):
    print(f"   {i:2d}. {c}")

## Step 4 · ตรวจสอบข้อมูลดิบ — ก่อน clean

ดู 5 แถวแรก แล้วดูค่าใน column `ED` (หมวดยา) ว่ามีหน้าตาเป็นยังไง

In [ ]:
# ดู 5 แถวแรก เฉพาะคอลัมน์สำคัญ
display(df_raw[["name1", "name2", "generic name", "dosage", "ED"]].head())

In [ ]:
# ดูการกระจายหมวด ED — สังเกตว่ามีค่าซ้ำเพราะ whitespace
print("🔍 หมวด ED ดิบ (มีปัญหา whitespace):")
print(df_raw["ED"].value_counts(dropna=False))

**ปัญหาที่เจอ:**
- `"ก"` กับ `" ก"` (มีช่องว่างหน้า) ถูกแยกกัน → ต้อง clean
- `"ง"` กับ `" ง"` ก็เหมือนกัน

ขั้นตอนต่อไปจะแก้ตรงนี้

### 🔬 ตรวจสอบความครบถ้วนของคอลัมน์ clinical

ก่อน clean ขอดูว่าคอลัมน์ clinical ทั้ง 4 (คำแนะนำ, เงื่อนไข, คำเตือน, หมายเหตุ) มีข้อมูลจริงเท่าไร — เพื่อเข้าใจคุณภาพข้อมูล

In [ ]:
clinical_cols = ["คำแนะนำ", "เงื่อนไข", "คำเตือนและข้อควรระวัง", "หมายเหตุ"]
print("📊 จำนวนแถวที่มีข้อมูล (ไม่ว่าง) ในแต่ละคอลัมน์:")
for c in clinical_cols:
    n = df_raw[c].notna().sum()
    pct = n / len(df_raw) * 100
    print(f"   {c:30s}: {n:4d} / {len(df_raw)} ({pct:5.1f}%)")

## Step 5 · Clean Step 1 — ลบช่องว่าง (whitespace)

ใช้ `.str.strip()` ลบช่องว่างหน้า/หลังของทุก string column รวมทั้งคอลัมน์ clinical 4 อัน

In [ ]:
df = df_raw.copy()  # copy ไม่แก้ของเดิม

# Strip whitespace ใน text columns (รวมคอลัมน์ clinical ทั้ง 4)
text_cols = ["ED", "name1", "name2", "generic name", "syn name",
             "dosage", "ประเภทยา",
             "คำแนะนำ", "เงื่อนไข", "คำเตือนและข้อควรระวัง", "หมายเหตุ",
             "Code"]
for c in text_cols:
    if c in df.columns:
        df[c] = df[c].astype(str).str.strip().replace("nan", np.nan)

print("✅ หลัง clean — หมวด ED:")
print(df["ED"].value_counts())

## Step 6 · เพิ่มความหมายของหมวด ED + สิทธิการเบิก

นี่คือ **value-add หลักของ Kade** — เปลี่ยนจากแค่ตัวอักษร ก/ข/ค/ง/จ(1)/จ(2) ให้เป็นข้อมูลที่คนใช้จริงเข้าใจได้

**ที่มา:** บัญชียาหลักแห่งชาติ — หมวด ED บอกว่าผู้ป่วยจะเบิกฟรีได้ไหม

In [ ]:
# Lookup table — ความหมาย + สิทธิเบิก + สี badge
ED_LOOKUP = {
    "ก":    ("ยาจำเป็นพื้นฐาน",        "บัตรทอง/ประกันสังคมเบิกได้ทุกกรณี",  "🟢"),
    "ข":    ("ยาที่ใช้ได้ตามเงื่อนไข",  "เบิกได้บางกรณี ตามเงื่อนไขที่กำหนด",  "🟡"),
    "ค":    ("ยาโรงพยาบาล",            "ต้องอยู่ในบัญชียาของ รพ.นั้น",         "🔵"),
    "ง":    ("ยาโรคเฉพาะทาง",          "ใช้กับโรคเฉพาะ ตามแนวทางการรักษา",   "🟣"),
    "จ(1)": ("ยาราคาแพง/ยาใหม่",        "ต้องขออนุมัติก่อนใช้",                "🟠"),
    "จ(2)": ("ยาพิเศษ",                "ต้องขออนุมัติ + เงื่อนไขพิเศษ",        "🔴"),
}

df["ed_meaning"]         = df["ED"].map(lambda x: ED_LOOKUP.get(x, ("—", "—", "⚪"))[0])
df["reimbursement_note"] = df["ED"].map(lambda x: ED_LOOKUP.get(x, ("—", "—", "⚪"))[1])
df["badge_color"]        = df["ED"].map(lambda x: ED_LOOKUP.get(x, ("—", "—", "⚪"))[2])

# ดูผลลัพธ์
display(df[["generic name", "ED", "ed_meaning", "reimbursement_note", "badge_color"]].head(10))

## Step 7 · Map กลุ่มยา → ICD-10 chapter

**เหตุผล:** ตารางของพี่ Getto (`disease_specialty_mapping.csv`) มีคอลัมน์ `icd10_chapter` (เช่น XI, IX, X) เพื่อให้แอปจับคู่ได้ระหว่าง "โรคที่แนะนำ" กับ "ยาที่เกี่ยวข้อง" เราต้อง map คอลัมน์ `name1` (Gastro-intestinal system, etc.) → ICD-10 chapter

**Source:** ICD-10 standard chapter classification (WHO)

In [ ]:
# Map drug body system group → ICD-10 chapter (Roman numeral) + Thai chapter name
ICD_CHAPTER_LOOKUP = {
    "Gastro-intestinal system":                            ("XI",   "โรคระบบทางเดินอาหาร"),
    "Cardiovascular system":                               ("IX",   "โรคระบบไหลเวียนโลหิต"),
    "Respiratory system":                                  ("X",    "โรคระบบทางเดินหายใจ"),
    "Central nervous system":                              ("VI",   "โรคระบบประสาท"),
    "Infections":                                          ("I",    "โรคติดเชื้อและปรสิต"),
    "Endocrine system":                                    ("IV",   "โรคต่อมไร้ท่อ โภชนาการ และเมตาบอลิซึม"),
    "Skin":                                                ("XII",  "โรคผิวหนังและเนื้อเยื่อใต้ผิวหนัง"),
    "Eye":                                                 ("VII",  "โรคตาและส่วนประกอบของตา"),
    "Ear, nose, oropharynx  and oral cavity":              ("VIII", "โรคหูและกระดูกกกหู"),
    "Musculoskeletal and joint diseases":                  ("XIII", "โรคระบบกล้ามเนื้อ กระดูก และเนื้อเยื่อเกี่ยวพัน"),
    "Obstetrics, gynaecology and urinary-tract disorders": ("XIV",  "โรคระบบสืบพันธุ์-ปัสสาวะ"),
    "Nutrition and blood":                                 ("III",  "โรคของเลือดและอวัยวะสร้างเลือด"),
    "Malignant disease and immunosuppression":             ("II",   "โรคเนื้องอก/มะเร็ง"),
    "Immunological products and vaccines":                 ("XXI",  "ปัจจัยที่ส่งผลต่อสุขภาพและการรับบริการ"),
    "Antidotes":                                           ("XIX",  "การบาดเจ็บ พิษ และเหตุภายนอก"),
    "Anesthesia":                                          ("XX",   "สาเหตุภายนอกของการป่วยและการตาย"),
    "Contrast media and Radiopharmaceuticals":             ("XXI",  "ปัจจัยที่ส่งผลต่อสุขภาพและการรับบริการ"),
}

df["icd10_chapter"]    = df["name1"].map(lambda x: ICD_CHAPTER_LOOKUP.get(x, ("—", "—"))[0])
df["icd10_chapter_th"] = df["name1"].map(lambda x: ICD_CHAPTER_LOOKUP.get(x, ("—", "—"))[1])

print("📊 จำนวนยาตาม ICD-10 chapter:")
print(df.groupby(["icd10_chapter", "icd10_chapter_th"]).size().sort_values(ascending=False))

## Step 8 · เลือกเฉพาะคอลัมน์ที่จะใช้ + เปลี่ยนชื่อให้อ่านง่าย

ตัดคอลัมน์ที่ไม่ได้ใช้ออก (เช่น `grcode1-4`, `link file`, `Footnote`) → **เก็บคอลัมน์ clinical ทั้ง 4** (คำแนะนำ, เงื่อนไข, คำเตือน, หมายเหตุ)

> 💡 หมายเหตุ: คอลัมน์ `recommendation_note` (คำแนะนำ) มีข้อมูลเพียง 3 รายการ แต่เก็บไว้เพราะเนื้อหาคุณภาพสูง (อ้างอิง WHO/UNICEF)

In [ ]:
# เลือกเฉพาะคอลัมน์ที่ต้องใช้ + rename ให้สื่อความหมาย
df_final = df[[
    "generic name", "syn name", "dosage", "ประเภทยา",
    "ED", "ed_meaning", "reimbursement_note", "badge_color",
    "name1", "name2",
    "icd10_chapter", "icd10_chapter_th",
    # ↓ Clinical info 4 columns
    "คำแนะนำ", "เงื่อนไข", "คำเตือนและข้อควรระวัง", "หมายเหตุ",
]].rename(columns={
    "generic name":          "generic_name",
    "syn name":              "syn_name",
    "ประเภทยา":              "drug_type",
    "ED":                    "ed_category",
    "name1":                 "drug_group_en",
    "name2":                 "drug_subgroup_en",
    "คำแนะนำ":               "recommendation_note",
    "เงื่อนไข":              "condition_note",
    "คำเตือนและข้อควรระวัง":   "warning_note",
    "หมายเหตุ":              "remark",
})

# เพิ่มคอลัมน์ source (ที่มาของข้อมูล)
df_final["source_name"] = "บัญชียาหลักแห่งชาติ พ.ศ. 2558"
df_final["source_url"]  = "http://drug.fda.moph.go.th:81/nlem.in.th/medicine/essential/list"
df_final["publisher"]   = "กลุ่มนโยบายแห่งชาติด้านยา สำนักยา กระทรวงสาธารณสุข"

print(f"✅ ขนาดสุดท้าย: {df_final.shape[0]} แถว × {df_final.shape[1]} คอลัมน์")
print(f"\n📋 คอลัมน์ที่ได้:")
for c in df_final.columns:
    print(f"   • {c}")

# สรุปความครบถ้วนของคอลัมน์ clinical
print(f"\n📊 ความครบถ้วนของคอลัมน์ clinical:")
for c in ["recommendation_note", "condition_note", "warning_note", "remark"]:
    n = df_final[c].notna().sum()
    print(f"   • {c}: {n} / {len(df_final)} ({n/len(df_final)*100:.1f}%)")

display(df_final.head(5))

## Step 9 · Save เป็น CSV

ใช้ `encoding="utf-8-sig"` เพื่อให้ Excel เปิดภาษาไทยได้ถูกต้อง

In [ ]:
OUTPUT_FILE = "drugs_clean.csv"
df_final.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")

print(f"✅ บันทึก {OUTPUT_FILE} สำเร็จ ({len(df_final)} แถว)")

# Verify ด้วยการอ่านกลับมา
df_check = pd.read_csv(OUTPUT_FILE, encoding="utf-8-sig")
print(f"✅ อ่านกลับได้: {df_check.shape}")
display(df_check.head(3))

## Step 10 · ดาวน์โหลดไฟล์ CSV

หลัง download → upload ไฟล์เข้า `data/processed/drugs_clean.csv` ใน GitHub repo (ลาก-วางผ่าน browser ได้เลย)

In [ ]:
from google.colab import files
files.download(OUTPUT_FILE)

---

## 🎯 สรุป

✅ อ่านไฟล์ต้นฉบับจากกระทรวงสาธารณสุข (831 รายการ)
✅ Clean whitespace + normalize หมวด ED
✅ เพิ่มความหมาย ก/ข/ค/ง/จ(1)/จ(2) + สิทธิการเบิก
✅ Map กลุ่มยา → ICD-10 chapter (เชื่อมกับฐาน 41 โรคของพี่ Getto)
✅ **เก็บข้อมูล clinical ครบ 4 คอลัมน์** (คำแนะนำ, เงื่อนไข, คำเตือน, หมายเหตุ)
✅ Save เป็น CSV พร้อมใช้ในแอป Streamlit

**Next step:** เอาไฟล์ `drugs_clean.csv` push ขึ้น MongoDB Atlas เพื่อใช้เป็น external database (ทำใน notebook ที่ 2 — `02_push_to_mongodb.ipynb`)